In [1]:
#librerias
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.stats import norm
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import os

C:\Users\HP\Documents\GitHub\TesisPEG\gpu_env_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================
# 0) Load CSV from Drive
# =========================
#drive.mount("/content/drive")
CSV_PATH = "../results/fechas_117/TERMICA/inputRed2.csv"   # <-- adjust
df = pd.read_csv(
    CSV_PATH,
    sep=";",
    decimal="."     # si tienes 1.234.567,89
)
df.columns = df.columns.str.strip() 
df=df.sort_values(["Fecha", "id_planta"]).reset_index(drop=True)
required = ['id_planta', "available_q", "phi", "precio_d", "daily_reman_demand"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# Optional: if present, enables per-player phi sampling
AGENT_COL = "id_planta" if "id_planta" in df.columns else None



In [3]:
df.columns

Index(['CodigoPlanta', 'available_q', 'phi', 'precio_d', 'Fecha', 'cluster',
       'id_planta', 'daily_reman_demand'],
      dtype='str')

In [4]:
import ast
import numpy as np
import pandas as pd

# =========================================================
# 1) Parser robusto para columnas que vienen como listas
# =========================================================
def _to_array24(x):
    if isinstance(x, np.ndarray):
        arr = x.astype(float).reshape(-1)
    elif isinstance(x, list):
        arr = np.asarray(x, dtype=float).reshape(-1)
    elif isinstance(x, str):
        arr = np.asarray(ast.literal_eval(x), dtype=float).reshape(-1)
    else:
        raise ValueError(f"No pude convertir a array: {type(x)}")

    if arr.size != 24:
        raise ValueError(f"Esperaba longitud 24 y llegó {arr.size}")
    return arr


def _safe_float(x, default=0.0):
    try:
        v = float(x)
        if np.isfinite(v):
            return v
        return default
    except:
        return default


# =========================================================
# 2) Construir juego para un solo día
# =========================================================
def build_juego_from_day(df_day, bid_col="est_uni"):
    """
    df_day: subconjunto de un solo día
    bid_col: columna que usarás como oferta/bid
             por ejemplo 'est_uni', 'est_dis' o 'precio_d'
    """
    df_day = df_day.copy()

    # demanda diaria de 24 horas: tomo la primera no nula
    demand_val = None
    for x in df_day["daily_reman_demand"]:
        if pd.notna(x):
            demand_val = x
            break
    if demand_val is None:
        raise ValueError("No encontré daily_reman_demand para este día")

    demanda24 = _to_array24(demand_val)

    juego = {
        "I": np.asarray(df_day["id_planta"].astype(int).tolist(), dtype=int),
        "demanda": demanda24,
    }

    meta_cols = [
        "CodigoPlanta", "phi", "precio_d", "Fecha", "cluster",
        "id_planta", "daily_reman_demand", "Grupo", "est_uni", "est_dis", 'Combustible por Defecto'
    ]

    for _, row in df_day.iterrows():
        pid = int(row["id_planta"])

        juego[pid] = {
            "beta": _safe_float(row[bid_col], default=0.0),
            "available_q": _to_array24(row["available_q"]),
        }

        for col in meta_cols:
            if col in row.index:
                juego[pid][col] = row[col]

    return juego


# =========================================================
# 3) Aplanar equilibrio a nivel planta-día
# =========================================================
def flatten_eq_daily(eq):
    rows = []

    pricing = eq["pricing"]
    price24_sys = eq["price24"]

    for pid in eq["I"]:
        pid = int(pid)
        x = eq[pid]

        q = np.asarray(x["cantidades_despachadas"], dtype=float).reshape(24)
        p = np.asarray(x["precio24"], dtype=float).reshape(24)

        row = {
            "Fecha": x.get("Fecha"),
            "id_planta": x.get("id_planta", pid),
            "CodigoPlanta": x.get("CodigoPlanta"),
            "Grupo": x.get("Grupo"),
            "cluster": x.get("cluster"),
            "phi": x.get("phi"),
            "precio_d": x.get("precio_d"),
            "est_uni": x.get("est_uni"),
            "est_dis": x.get("est_dis"),
            "bid": x["beta"],
            "pricing": pricing,
            "cantidades_despachadas": q.tolist(),
            "precio24": p.tolist(),
            "price24_sistema": np.asarray(price24_sys, dtype=float).tolist(),
            "sumaq": float(q.sum()),
            "ingreso_total": float((q * p).sum()),
            'Combustible': x.get("Combustible por Defecto")
        }
        rows.append(row)

    out = pd.DataFrame(rows)
    if "Fecha" in out.columns:
        out["Fecha"] = pd.to_datetime(out["Fecha"])
    return out


# =========================================================
# 4) Expandir a nivel planta-día-hora
# =========================================================
def explode_dispatch_hourly(df_daily_eq):
    rows = []

    for _, row in df_daily_eq.iterrows():
        q = np.asarray(row["cantidades_despachadas"], dtype=float).reshape(24)
        p = np.asarray(row["precio24"], dtype=float).reshape(24)
        ps = np.asarray(row["price24_sistema"], dtype=float).reshape(24)

        for h in range(24):
            rows.append({
                "Fecha": row["Fecha"],
                "hora": h,
                "id_planta": row["id_planta"],
                "CodigoPlanta": row["CodigoPlanta"],
                "Grupo": row["Grupo"],
                "cluster": row["cluster"],
                "phi": row["phi"],
                "pricing": row["pricing"],
                "bid": row["bid"],
                "cantidad_despachada": float(q[h]),
                "precio_pagado": float(p[h]),
                "precio_sistema": float(ps[h]),
                "ingreso_hora": float(q[h] * p[h])
            })

    out = pd.DataFrame(rows)
    if "Fecha" in out.columns:
        out["Fecha"] = pd.to_datetime(out["Fecha"])
    return out

import numpy as np

def compute_equilibrio_dispatch(juego: dict, pricing: str = "uniform", unmet_price="max_accepted"):
    """
    unmet_price:
      - número (float): usa ese precio si no hay dispatch
      - "max_accepted": si no hay dispatch, usa max bid positivo del sistema (si existe; si no, 0)
    """
    pricing = pricing.lower().strip()
    if pricing not in {"uniform", "pay_as_bid"}:
        raise ValueError("pricing must be 'uniform' or 'pay_as_bid'")

    # ids
    if "I" in juego:
        plant_ids = [int(x) for x in np.asarray(juego["I"]).reshape(-1)]
    else:
        plant_ids = [k for k in juego.keys() if isinstance(k, (int, np.integer))]

    if len(plant_ids) == 0:
        raise ValueError("No plant ids found in juego")

    demand24 = np.asarray(juego["demanda"], dtype=float).reshape(24)
    bids = np.array([float(juego[pid]["beta"]) for pid in plant_ids], dtype=float)
    avail_mat = np.vstack([
        np.asarray(juego[pid]["available_q"], dtype=float).reshape(24)
        for pid in plant_ids
    ])

    I = len(plant_ids)
    disp_mat = np.zeros((I, 24), dtype=float)
    price24 = np.zeros(24, dtype=float)

    # merit order: solo bids positivos
    active = bids > 0
    order = np.array([], dtype=int)
    if np.any(active):
        order = np.where(active)[0][np.argsort(bids[active], kind="mergesort")]

    max_bid_pos = float(np.max(bids[active])) if np.any(active) else 0.0

    for h in range(24):
        remaining = float(max(demand24[h], 0.0))
        if remaining <= 0:
            price24[h] = 0.0
            continue

        av_h = np.maximum(avail_mat[:, h], 0.0)
        last_bid = None
        any_dispatch = False

        for i in order:
            if remaining <= 0:
                break
            take = min(av_h[i], remaining)
            if take > 0:
                disp_mat[i, h] = take
                remaining -= take
                any_dispatch = True
                last_bid = float(bids[i])

        if any_dispatch:
            price24[h] = float(last_bid)
        else:
            if unmet_price == "max_accepted":
                price24[h] = max_bid_pos
            else:
                price24[h] = float(unmet_price)

    if pricing == "uniform":
        precio_mat = np.where(disp_mat > 0.0, price24.reshape(1, 24), 0.0)
    else:
        precio_mat = np.where(disp_mat > 0.0, bids.reshape(I, 1), 0.0)

    eq = {
        "I": np.array(plant_ids, dtype=int),
        "demanda": demand24.copy(),
        "pricing": pricing,
        "price24": price24.copy(),
    }

    meta_cols = [
        "CodigoPlanta", "phi", "precio_d", "Fecha", "cluster",
        "id_planta", "daily_reman_demand", "Grupo", "est_uni", "est_dis", 'Combustible por Defecto'
    ]

    for idx, pid in enumerate(plant_ids):
        q = disp_mat[idx, :].copy()

        eq[pid] = {
            "beta": float(bids[idx]),
            "available_q": avail_mat[idx, :].copy(),
            "cantidades_despachadas": q,
            "precio24": precio_mat[idx, :].copy(),
            "sumaq": float(q.sum()),
        }

        if isinstance(juego.get(pid, {}), dict):
            for col in meta_cols:
                if col in juego[pid]:
                    val = juego[pid][col]
                    if isinstance(val, (list, tuple, np.ndarray)):
                        eq[pid][col] = np.asarray(val).copy()
                    else:
                        eq[pid][col] = val

        if "id_planta" not in eq[pid]:
            eq[pid]["id_planta"] = int(pid)

    return eq
# =========================================================
# 5) Correr todo el panel por fecha
# =========================================================
def compute_dispatch_panel(df, bid_col="est_uni", pricing="uniform", unmet_price="max_accepted"):
    """
    df: tu base completa con muchas fechas
    bid_col:
        - "est_uni"  para usar los bids estimados bajo uniforme
        - "est_dis"  para usar los bids estimados bajo discriminatorio
        - "precio_d" si quieres usar esa columna como bid
    pricing:
        - "uniform"
        - "pay_as_bid"
    """
    df = df.copy()
    df["Fecha"] = pd.to_datetime(df["Fecha"])

    daily_results = []

    for fecha, g in df.sort_values(["Fecha", "id_planta"]).groupby("Fecha", sort=True):
        juego = build_juego_from_day(g, bid_col=bid_col)
        eq = compute_equilibrio_dispatch(
            juego=juego,
            pricing=pricing,
            unmet_price=unmet_price
        )
        df_eq_day = flatten_eq_daily(eq)
        daily_results.append(df_eq_day)

    df_daily_eq = pd.concat(daily_results, ignore_index=True)
    df_hourly_eq = explode_dispatch_hourly(df_daily_eq)

    return df_daily_eq, df_hourly_eq

## Computar Q min, lo de Q tilde

In [5]:
df_desglose = pd.read_excel("../datos/raw/desglose.xlsx",header=3) #cambiar la direccion
df_desglose

,Código SIC,Nombre Recurso,Capacidad Efectiva Neta [MW],Factor de Conversión,Es Menor,Tipo Despacho,Combustible por Defecto,Fecha Operación,Municipio,Departamento,Agente Representante,Estado Recurso,Tipo Generación,Clasificación
0,5E69,TERMOSARAVENA 1,4.9,10.7650,SI,ND,GAS,2025-10-31,SARAVENA,ARAUCA,GENERSA S.A.S. E.S.P.,PRUEBAS,TERMICA,AUTOGENERADOR
1,4YCN,VILLANUEVA 1,25.0,11.9149,NO,DC,BIOMASA,2025-10-02,VILLANUEVA,CASANARE,REFOENERGY VILLANUEVA S.A.S E.S.P.,OPERACIÓN,TERMICA,NORMAL
2,4XP3,AUTOG TERMOARAUCA 1,4.9,10.7650,NO,ND,GAS,2025-08-27,SARAVENA,ARAUCA,GENERSA S.A.S. E.S.P.,OPERACIÓN,TERMICA,AUTOGENERADOR
3,4T6C,AUTOG LOS POCITOS 1,4.1,4.3900,NO,ND,BIOGAS,2025-02-16,TUBARA,ATLÁNTICO,BCCY CORDOBA S.A.S. E.S.P.,OPERACIÓN,TERMICA,AUTOGENERADOR
4,3NNZ,AUTOG TERMOCOROCITO 1,4.9,0.0000,NO,ND,GAS,2024-10-29,TAME,ARAUCA,GENERSA S.A.S. E.S.P.,OPERACIÓN,TERMICA,AUTOGENERADOR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502,ILC1,INGENIO LA CARMELITA,0.0,0.0000,NO,ND,BAGAZO,2009-11-11,TULUA,VALLE DEL CAUCA,DEPI ENERGY S.A.S. E.S.P.,INACTIVO,COGENERADOR,COGENERADOR
503,CIP1,INGENIO PROVIDENCIA 2,19.9,0.0000,NO,ND,CARBON,2009-05-18,EL CERRITO,VALLE DEL CAUCA,NITRO ENERGY COLOMBIA S.A.S. E.S.P.,OPERACIÓN,COGENERADOR,COGENERADOR
504,PPN1,PAPELES NACIONALES,0.0,0.0000,NO,ND,GAS,2009-02-05,PEREIRA,RISARALDA,ENERCO S.A. E.S.P.,OPERACIÓN,COGENERADOR,COGENERADOR
505,CAS1,CENTRAL CASTILLA 1,3.0,0.0000,NO,ND,BAGAZO,2004-08-15,PRADERA,VALLE DEL CAUCA,RIOPAILA ENERGÍA S.A.S. E.S.P.,OPERACIÓN,COGENERADOR,COGENERADOR


In [6]:

df_desglose['Código SIC'] = df_desglose['Código SIC'].astype(str).str.strip()
df['CodigoPlanta'] = df['CodigoPlanta'].astype(str).str.strip()
df = df.merge(
    df_desglose[["Código SIC", 'Combustible por Defecto', 'Nombre Recurso', 'Agente Representante']],
    left_on="CodigoPlanta",
    right_on="Código SIC",
    how="left"
)
df = df.drop(columns=["Código SIC"])

In [7]:
df.columns

Index(['CodigoPlanta', 'available_q', 'phi', 'precio_d', 'Fecha', 'cluster',
       'id_planta', 'daily_reman_demand', 'Combustible por Defecto',
       'Nombre Recurso', 'Agente Representante'],
      dtype='str')

# leer carpeta

In [8]:
agentes_permitidos = df['Agente Representante'].unique().tolist()
agentes_permitidos

['TERMOYOPAL GENERACION 2 S.A.S E.S.P.',
 'TERMOTASAJERO DOS S.A. E.S.P. ',
 'TERMOTASAJERO S.A. E.S.P.',
 'CELSIA COLOMBIA S.A. E.S.P.',
 'GESTION ENERGETICA S.A. E.S.P.',
 'GENERADORA Y COMERCIALIZADORA DE ENERGIA DEL CARIBE S.A. E.S.P.',
 'COMPAÑÍA ELÉCTRICA DE SOCHAGOTA  S.A.S. E.S.P.',
 'ENEL COLOMBIA SA ESP',
 'EMPRESAS PUBLICAS DE MEDELLIN E.S.P.',
 'GENERADORA TERMOCENTRO S.A.S. E.S.P.',
 'ENFRAGEN TERMOFLORES S.A.S. E.S.P.',
 'PROELECTRICA S.A.S E.S.P.',
 'NITRO ENERGY COLOMBIA S.A.S. E.S.P.',
 'TERMOCANDELARIA S.A.S. - E.S.P. ',
 'TERMOBARRANQUILLA S.A. EMPRESA DE SERVICIOS PUBLICOS',
 'TERMONORTE S.A.S. E.S.P.',
 'ENFRAGEN TERMOVALLE S.A.S. EMPRESA DE SERVICIOS PUBLICOS',
 'TERMOEMCALI I S.A. E.S.P.']

In [9]:
import glob
import os

# Ruta recursiva para buscar todos los archivos .txt dentro de MO
ruta_base = "../datos/raw/MO/**/*.txt"
archivos = glob.glob(ruta_base, recursive=True)

dict_mo = {}

print(f"🚀 Procesando {len(archivos)} archivos. Extrayendo todos los MO encontrados...")

for archivo in archivos:
    try:
        nombre_archivo = os.path.basename(archivo)
        # Extraer fecha (asumiendo formato OFEImmdd.txt)
        fecha_str = f"2025-{nombre_archivo[4:6]}-{nombre_archivo[6:8]}"
        
        datos_dia = []
        
        with open(archivo, 'r', encoding='latin-1') as f:
            for linea in f:
                # Limpiar y separar por coma
                partes = [p.strip() for p in linea.split(',')]
                
                # Buscamos filas donde el segundo elemento sea 'MO'
                # Y que la línea tenga suficientes columnas para los valores horarios
                if len(partes) >= 26 and partes[1] == 'MO':
                    recurso = partes[0]
                    # Convertimos las horas a float, manejando posibles errores de formato
                    try:
                        valores = [float(x) for x in partes[2:26]]
                        datos_dia.append((recurso, valores))
                    except ValueError:
                        continue
        
        if datos_dia:
            if fecha_str not in dict_mo:
                dict_mo[fecha_str] = {}
            # Guardamos todo lo encontrado
            dict_mo[fecha_str].update({recurso: valores for recurso, valores in datos_dia})
            
    except Exception as e:
        print(f"❌ Error en {archivo}: {e}")

print(f"✅ Extracción completada. Fechas procesadas: {len(dict_mo)}")

🚀 Procesando 186 archivos. Extrayendo todos los MO encontrados...
✅ Extracción completada. Fechas procesadas: 183


In [10]:
import pandas as pd

# Convertir el diccionario directamente a una lista de registros compactos
data_compacta = []

for fecha, recursos in dict_mo.items():
    for nombre_recurso, valores in recursos.items():
        data_compacta.append({
            'Fecha': fecha,
            'Nombre Recurso': nombre_recurso,
            'MO_array': valores  # Aquí guardas la lista de 24 elementos completa
        })

# Crear el DataFrame
df_mo_compacto = pd.DataFrame(data_compacta)

# Asegurar tipos de datos para el merge posterior
df_mo_compacto['Fecha'] = pd.to_datetime(df_mo_compacto['Fecha'])

# Si tu DF principal tiene una columna 'Fecha' diferente, asegúrala también
# df['Fecha'] = pd.to_datetime(df['Fecha'])

print("✅ DataFrame creado con estructura compacta:")
print(df_mo_compacto.head())

✅ DataFrame creado con estructura compacta:
       Fecha Nombre Recurso                                           MO_array
0 2025-04-01      SALVAJINA  [155.0, 155.0, 155.0, 155.0, 155.0, 155.0, 201...
1 2025-04-01          ALBAN  [9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, ...
2 2025-04-01        BETANIA  [60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60....
3 2025-04-01       ELQUIMBO  [86.0, 86.0, 86.0, 86.0, 86.0, 86.0, 86.0, 86....
4 2025-04-01        ITUANGO  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...


In [11]:
df_mo_compacto

,Fecha,Nombre Recurso,MO_array
0,2025-04-01,SALVAJINA,"[155.0, 155.0, 155.0, 155.0, 155.0, 155.0, 201..."
1,2025-04-01,ALBAN,"[9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, ..."
2,2025-04-01,BETANIA,"[60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60...."
3,2025-04-01,ELQUIMBO,"[86.0, 86.0, 86.0, 86.0, 86.0, 86.0, 86.0, 86...."
4,2025-04-01,ITUANGO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...
1459,2025-09-30,ELQUIMBO,"[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100..."
1460,2025-09-30,ITUANGO,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1461,2025-09-30,MIELI,"[31.0, 31.0, 31.0, 31.0, 31.0, 31.0, 31.0, 31...."
1462,2025-09-30,SOGAMOSO,"[120.0, 120.0, 120.0, 120.0, 120.0, 120.0, 120..."


In [12]:
import pandas as pd
import unicodedata

def limpiar_nombre(texto):
    """
    1. Convierte a minúsculas.
    2. Elimina tildes (normaliza a NFD y descarta caracteres no espaciados).
    3. Elimina espacios en blanco laterales.
    """
    texto = str(texto).lower().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto

# Aplicamos la limpieza a tu DataFrame principal (df)
# Asegúrate de que el nombre de la columna sea el correcto (ej: 'id_planta' o 'nombre_recurso')
df['Nombre Recurso'] = df['Nombre Recurso'].apply(limpiar_nombre)

# Aplicamos la limpieza al DataFrame del MO
df_mo_compacto['Nombre Recurso'] = df_mo_compacto['Nombre Recurso'].apply(limpiar_nombre)
df['Fecha']=pd.to_datetime(df['Fecha'])
df_mo_compacto['Fecha']=pd.to_datetime(df_mo_compacto['Fecha'])

# Ahora el merge debería ser perfecto y sin errores por formato
df_final = pd.merge(
    df, 
    df_mo_compacto, 
    on=['Fecha', 'Nombre Recurso'], 
    how='left'
)

In [13]:
df_mo_compacto

,Fecha,Nombre Recurso,MO_array
0,2025-04-01,salvajina,"[155.0, 155.0, 155.0, 155.0, 155.0, 155.0, 201..."
1,2025-04-01,alban,"[9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, 9.0, ..."
2,2025-04-01,betania,"[60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 60...."
3,2025-04-01,elquimbo,"[86.0, 86.0, 86.0, 86.0, 86.0, 86.0, 86.0, 86...."
4,2025-04-01,ituango,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...
1459,2025-09-30,elquimbo,"[100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100..."
1460,2025-09-30,ituango,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1461,2025-09-30,mieli,"[31.0, 31.0, 31.0, 31.0, 31.0, 31.0, 31.0, 31...."
1462,2025-09-30,sogamoso,"[120.0, 120.0, 120.0, 120.0, 120.0, 120.0, 120..."


In [14]:
df

,CodigoPlanta,available_q,phi,precio_d,Fecha,cluster,id_planta,daily_reman_demand,Combustible por Defecto,Nombre Recurso,Agente Representante
0,TYP3,"[49000.0, 49000.0, 49000.0, 49000.0, 49000.0,...",83.924097,101.37,2025-04-01,10.0,1,"[1029700.0, 935380.0, 843320.0, 771690.0, 695...",GAS,termoyopal g3,TERMOYOPAL GENERACION 2 S.A.S E.S.P.
1,TYP2,"[26000.0, 26000.0, 26000.0, 26000.0, 26000.0,...",262.221847,282.95,2025-04-01,,2,"[1029700.0, 935380.0, 843320.0, 771690.0, 695...",GAS,termoyopal 2,TERMOYOPAL GENERACION 2 S.A.S E.S.P.
2,TSJ2,"[170000.0, 170000.0, 170000.0, 170000.0, 1700...",275.288434,326.37,2025-04-01,10.0,3,"[1029700.0, 935380.0, 843320.0, 771690.0, 695...",CARBON,tasajero 2,TERMOTASAJERO DOS S.A. E.S.P.
3,TSJ1,"[165000.0, 165000.0, 165000.0, 165000.0, 1650...",260.379933,310.37,2025-04-01,10.0,4,"[1029700.0, 935380.0, 843320.0, 771690.0, 695...",CARBON,tasajero 1,TERMOTASAJERO S.A. E.S.P.
4,MRL1,"[164000.0, 164000.0, 164000.0, 164000.0, 1640...",968.472039,1028.13,2025-04-01,10.0,5,"[1029700.0, 935380.0, 843320.0, 771690.0, 695...",GAS,merilectrica 1,CELSIA COLOMBIA S.A. E.S.P.
...,...,...,...,...,...,...,...,...,...,...,...
4207,TRN1,"[80000.0, 80000.0, 80000.0, 80000.0, 80000.0,...",779.362162,908.94,2025-09-09,12.0,32,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80...",GAS,termonorte,TERMONORTE S.A.S. E.S.P.
4208,TBQ4,"[60000.0, 60000.0, 60000.0, 60000.0, 60000.0,...",1058.580711,1117.35,2025-09-09,12.0,33,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80...",GAS,barranquilla 4,TERMOBARRANQUILLA S.A. EMPRESA DE SERVICIOS PU...
4209,TBQ3,"[60000.0, 60000.0, 60000.0, 60000.0, 60000.0,...",945.489357,1012.74,2025-09-09,12.0,34,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80...",GAS,barranquilla 3,TERMOBARRANQUILLA S.A. EMPRESA DE SERVICIOS PU...
4210,TVL1,"[241000.0, 241000.0, 241000.0, 241000.0, 2410...",969.785873,1183.48,2025-09-09,12.0,35,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80...",ACPM,termovalle cc,ENFRAGEN TERMOVALLE S.A.S. EMPRESA DE SERVICIO...


In [15]:
##El MO es el piso técnico (el mínimo nivel de operación para mantenerse estable, especialmente en térmicas). Por lo tanto:Rango de operación: Entre $[MO, D]$.Zona de flexibilidad: $[0, D - MO]$.

In [16]:
df['Nombre Recurso'].unique()

<StringArray>
[     'termoyopal g3',       'termoyopal 2',         'tasajero 2',
         'tasajero 1',     'merilectrica 1',            'paipa 1',
       'termoyopal 1',      'termoyopal g4',         'gecelca 32',
            'paipa 4',            'paipa 2',            'paipa 3',
          'gecelca 3',          'zipaemg 5',          'zipaemg 2',
     'termosierra cc',          'zipaemg 3',     'termocentro cc',
          'guajira 1',          'guajira 2',        'flores i cc',
        'flores 4 cc',     'proelectrica 2',     'proelectrica 1',
      'termodorada 1',          'zipaemg 4',        'cartagena 1',
        'cartagena 2',           'tesorito', 'termocandelaria cc',
          'tebsab cc',         'termonorte',     'barranquilla 4',
     'barranquilla 3',      'termovalle cc',     'termoemcali cc']
Length: 36, dtype: str

In [17]:
df.columns

Index(['CodigoPlanta', 'available_q', 'phi', 'precio_d', 'Fecha', 'cluster',
       'id_planta', 'daily_reman_demand', 'Combustible por Defecto',
       'Nombre Recurso', 'Agente Representante'],
      dtype='str')

In [18]:
# Realizamos el merge. 
# Si tu df_daily_eq no tiene 'Hora', es posible que debas promediar 
# el MO a nivel diario antes del merge.

# Cruce final:
df = pd.merge(
    df, 
    df_mo_compacto, 
    left_on=['Fecha', 'Nombre Recurso'], 
    right_on=['Fecha', 'Nombre Recurso'], 
    how='left'
)

In [19]:
df.columns

Index(['CodigoPlanta', 'available_q', 'phi', 'precio_d', 'Fecha', 'cluster',
       'id_planta', 'daily_reman_demand', 'Combustible por Defecto',
       'Nombre Recurso', 'Agente Representante', 'MO_array'],
      dtype='str')

In [20]:
df_daily_eq, df_hourly_eq = compute_dispatch_panel(
    df,
    bid_col="precio_d",
    pricing="uniform",
    unmet_price="max_accepted"
)

In [21]:
df_daily_eq.columns

Index(['Fecha', 'id_planta', 'CodigoPlanta', 'Grupo', 'cluster', 'phi',
       'precio_d', 'est_uni', 'est_dis', 'bid', 'pricing',
       'cantidades_despachadas', 'precio24', 'price24_sistema', 'sumaq',
       'ingreso_total', 'Combustible'],
      dtype='str')

In [22]:
df_daily_eq

,Fecha,id_planta,CodigoPlanta,Grupo,cluster,phi,precio_d,est_uni,est_dis,bid,pricing,cantidades_despachadas,precio24,price24_sistema,sumaq,ingreso_total,Combustible
0,2025-04-01,1,TYP3,None,10.0,83.924097,101.37,None,None,101.37,uniform,"[49000.0, 49000.0, 49000.0, 49000.0, 49000.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",1176000.00,3.395024e+08,GAS
1,2025-04-01,2,TYP2,None,,262.221847,282.95,None,None,282.95,uniform,"[26000.0, 26000.0, 26000.0, 26000.0, 26000.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",259560.00,7.446551e+07,GAS
2,2025-04-01,3,TSJ2,None,10.0,275.288434,326.37,None,None,326.37,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",0.00,0.000000e+00,CARBON
3,2025-04-01,4,TSJ1,None,10.0,260.379933,310.37,None,None,310.37,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",36758.38,1.140870e+07,CARBON
4,2025-04-01,5,MRL1,None,10.0,968.472039,1028.13,None,None,1028.13,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",0.00,0.000000e+00,GAS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4207,2025-09-09,32,TRN1,None,12.0,779.362162,908.94,None,None,908.94,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS
4208,2025-09-09,33,TBQ4,None,12.0,1058.580711,1117.35,None,None,1117.35,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS
4209,2025-09-09,34,TBQ3,None,12.0,945.489357,1012.74,None,None,1012.74,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS
4210,2025-09-09,35,TVL1,None,12.0,969.785873,1183.48,None,None,1183.48,uniform,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,ACPM


## Calcular variables

In [23]:
import numpy as np
import pandas as pd

def _as_array24(x):
    arr = np.asarray(x, dtype=float).reshape(-1)
    if arr.size != 24:
        raise ValueError(f"Se esperaba un vector de 24 horas y llegó uno de tamaño {arr.size}")
    return arr

def add_S_columns_from_daily_eq(df_daily_eq, qmin_col=None, qmin_map=None):
    df = df_daily_eq.copy()

    if "cantidades_despachadas" not in df.columns:
        raise KeyError("Falta la columna 'cantidades_despachadas'")

    if qmin_col is None and qmin_map is None:
        df["_qmin"] = 0.0
    elif qmin_col is not None:
        if qmin_col not in df.columns:
            raise KeyError(f"No existe la columna {qmin_col}")
        df["_qmin"] = pd.to_numeric(df[qmin_col], errors="coerce").fillna(0.0)
    else:
        df["_qmin"] = df["id_planta"].map(qmin_map).fillna(0.0)

    S1_list, S2_list = [], []
    S3_list, S3_abs_list, S3_vector_list = [], [], []

    for _, row in df.iterrows():
        q = _as_array24(row["cantidades_despachadas"])
        qmin = float(row["_qmin"])

        # \tilde q_{iht} = max(q_{iht} - qmin_i, 0)
        q_tilde = np.maximum(q - qmin, 0.0)

        # Bordes:
        # h=0  => q_{h-1} = q_h
        # h=23 => q_{h+1} = q_h
        q_prev = np.r_[q[0], q[:-1]]
        q_next = np.r_[q[1:], q[-1]]

        # Vector horario:
        # (q_h - q_{h-1}) + (q_h - q_{h+1})
        s3_terms = (q - q_prev) + (q - q_next)

        S1_list.append(1.0)
        S2_list.append(float(q_tilde.sum()))

        # Esto siempre tenderá a cancelar y puede dar 0
        S3_list.append(float(s3_terms.sum()))

        # Esto mide magnitud de variación y no se cancela
        S3_abs_list.append(float(np.abs(s3_terms).sum()))

        # Guardar el vector horario completo
        S3_vector_list.append(s3_terms.tolist())

    df["S1_it"] = S1_list
    df["S2_it"] = S2_list
    df["S3_abs_it"] = S3_abs_list
    df["S3_vector_it"] = S3_vector_list

    df = df.drop(columns=["_qmin"])
    return df

In [24]:
import numpy as np

df_daily_eq['qmin'] = [np.zeros(24) for _ in range(len(df_daily_eq))]

In [25]:
df_daily_eq = add_S_columns_from_daily_eq(df_daily_eq, qmin_col="qmin")

In [26]:
df_daily_eq.to_csv('eq.csv')

In [27]:
df_daily_eq

,Fecha,id_planta,CodigoPlanta,Grupo,cluster,phi,precio_d,est_uni,est_dis,bid,...,precio24,price24_sistema,sumaq,ingreso_total,Combustible,qmin,S1_it,S2_it,S3_abs_it,S3_vector_it
0,2025-04-01,1,TYP3,None,10.0,83.924097,101.37,None,None,101.37,...,"[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",1176000.00,3.395024e+08,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,1176000.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,2025-04-01,2,TYP2,None,,262.221847,282.95,None,None,282.95,...,"[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",259560.00,7.446551e+07,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,259560.00,51120.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 440.0..."
2,2025-04-01,3,TSJ2,None,10.0,275.288434,326.37,None,None,326.37,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",0.00,0.000000e+00,CARBON,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,2025-04-01,4,TSJ1,None,10.0,260.379933,310.37,None,None,310.37,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",36758.38,1.140870e+07,CARBON,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,36758.38,147033.52,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,2025-04-01,5,MRL1,None,10.0,968.472039,1028.13,None,None,1028.13,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4207,2025-09-09,32,TRN1,None,12.0,779.362162,908.94,None,None,908.94,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4208,2025-09-09,33,TBQ4,None,12.0,1058.580711,1117.35,None,None,1117.35,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4209,2025-09-09,34,TBQ3,None,12.0,945.489357,1012.74,None,None,1012.74,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4210,2025-09-09,35,TVL1,None,12.0,969.785873,1183.48,None,None,1183.48,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,ACPM,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


## Regresion eq 19

In [28]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. Aseguramos que tenemos las variables necesarias
# (phi, S1_it, S2_it, S3_abs_it)
plantas = df_daily_eq['id_planta'].unique()
resultados_list = []

print(f"🚀 Iniciando regresiones individuales para {len(plantas)} plantas...")

for planta in plantas:
    # Filtramos los datos para esta planta
    df_planta = df_daily_eq[df_daily_eq['id_planta'] == planta].dropna()
    
    # Solo corremos si tenemos suficientes datos (ej. al menos 5 días)
    if len(df_planta) > 5:
        y = df_planta['phi']
        X = df_planta[['S1_it', 'S2_it', 'S3_abs_it']]
        X = sm.add_constant(X) # Agregamos el intercepto para cada planta
        
        # Ajustamos el modelo OLS
        model = sm.OLS(y, X).fit()
        
        # Guardamos los coeficientes
        resultados_list.append({
            'id_planta': planta,
            'intercepto': model.params['const'],
            'beta_S1': model.params['S1_it'],
            'beta_S2': model.params['S2_it'],
            'beta_S3': model.params['S3_abs_it'],
            'R2': model.rsquared,
            'p_value_S1': model.pvalues['S1_it']
        })

# 2. Creamos un DataFrame con los resultados
df_resultados = pd.DataFrame(resultados_list)

print("✅ Regresiones completadas.")
print(df_resultados.head())

🚀 Iniciando regresiones individuales para 36 plantas...
✅ Regresiones completadas.
Empty DataFrame
Columns: []
Index: []


In [29]:
# Verifica si hay datos
print(f"Total de filas en df_daily_eq: {len(df_daily_eq)}")
print(f"Plantas únicas detectadas: {df_daily_eq['id_planta'].unique()}")

# Prueba un filtro manual
ejemplo_planta = df_daily_eq['id_planta'].iloc[0]
test = df_daily_eq[df_daily_eq['id_planta'] == ejemplo_planta]
print(f"Filas para {ejemplo_planta}: {len(test)}")

Total de filas en df_daily_eq: 4212
Plantas únicas detectadas: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36]
Filas para 1: 117


## marginal costs

In [30]:
df_daily_eq

,Fecha,id_planta,CodigoPlanta,Grupo,cluster,phi,precio_d,est_uni,est_dis,bid,...,precio24,price24_sistema,sumaq,ingreso_total,Combustible,qmin,S1_it,S2_it,S3_abs_it,S3_vector_it
0,2025-04-01,1,TYP3,None,10.0,83.924097,101.37,None,None,101.37,...,"[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",1176000.00,3.395024e+08,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,1176000.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,2025-04-01,2,TYP2,None,,262.221847,282.95,None,None,282.95,...,"[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",259560.00,7.446551e+07,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,259560.00,51120.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 440.0..."
2,2025-04-01,3,TSJ2,None,10.0,275.288434,326.37,None,None,326.37,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",0.00,0.000000e+00,CARBON,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,2025-04-01,4,TSJ1,None,10.0,260.379933,310.37,None,None,310.37,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",36758.38,1.140870e+07,CARBON,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,36758.38,147033.52,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,2025-04-01,5,MRL1,None,10.0,968.472039,1028.13,None,None,1028.13,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[288.95, 288.92, 288.92, 288.92, 285.53, 285.5...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4207,2025-09-09,32,TRN1,None,12.0,779.362162,908.94,None,None,908.94,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4208,2025-09-09,33,TBQ4,None,12.0,1058.580711,1117.35,None,None,1117.35,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4209,2025-09-09,34,TBQ3,None,12.0,945.489357,1012.74,None,None,1012.74,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,GAS,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4210,2025-09-09,35,TVL1,None,12.0,969.785873,1183.48,None,None,1183.48,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[338.94, 289.45, 289.45, 288.95, 288.95, 288.9...",0.00,0.000000e+00,ACPM,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [31]:
import pandas as pd
import statsmodels.api as sm

# 1. LIMPIEZA OBLIGATORIA: Asegurar que los IDs sean idénticos
# Eliminamos espacios en blanco, convertimos a minúsculas y limpiamos tildes/caracteres raros
def limpiar_id(nombre):
    return str(nombre).strip().lower().replace(" ", "")

df_daily_eq['id_planta'] = df_daily_eq['id_planta'].apply(limpiar_id)

# 2. DEFINIR ESTRUCTURA DE RESULTADOS
lista_modelos = []

# 3. BUCLE POR CADA PLANTA
plantas = df_daily_eq['id_planta'].unique()

for planta in plantas:
    df_p = df_daily_eq[df_daily_eq['id_planta'] == planta].copy()
    
    # Filtramos filas con datos faltantes en las variables clave
    df_p = df_p.dropna(subset=['phi', 'S1_it', 'S2_it', 'S3_abs_it'])
    
    # Solo correr si tenemos suficientes datos para estimar (ej. al menos 10 días)
    if len(df_p) >= 10:
        y = df_p['phi']
        X = df_p[['S1_it', 'S2_it', 'S3_abs_it']]
        X = sm.add_constant(X)  # <--- AQUÍ ESTÁ TU CONSTANTE ÚNICA POR PLANTA
        
        modelo = sm.OLS(y, X).fit()
        
        # Guardamos parámetros
        lista_modelos.append({
            'id_planta': planta,
            'beta_S1': modelo.params['S1_it'],
            'beta_S2': modelo.params['S2_it'],
            'beta_S3': modelo.params['S3_abs_it'],
            'r_squared': modelo.rsquared
        })

# 4. CREAR TABLA FINAL CON LOS MODELOS
df_resultados_individuales = pd.DataFrame(lista_modelos)

print("✅ Regresiones individuales completadas con éxito.")
print(df_resultados_individuales.head())

✅ Regresiones individuales completadas con éxito.
  id_planta     beta_S1   beta_S2   beta_S3  r_squared
0         1  903.832288 -0.000719 -0.003281   0.644879
1         2  273.398803 -0.000176  0.000136   0.120349
2         3  694.377788 -0.000133 -0.000613   0.327912
3         4  532.990992 -0.000107 -0.000300   0.280324
4         5  835.396290 -0.112775 -0.225551   0.362880


In [32]:
df_resultados_individuales=df_resultados_individuales.merge(df_daily_eq[['id_planta', 'Combustible']].drop_duplicates(), on='id_planta')

## estos coeficientes cambian por dia

In [33]:
df_resultados_individuales

,id_planta,beta_S1,beta_S2,beta_S3,r_squared,Combustible
0,1,903.832288,-0.000719,-0.003281,6.448794e-01,GAS
1,2,273.398803,-0.000176,0.000136,1.203486e-01,GAS
2,3,694.377788,-0.000133,-0.000613,3.279121e-01,CARBON
3,4,532.990992,-0.000107,-0.000300,2.803243e-01,CARBON
4,5,835.396290,-0.112775,-0.225551,3.628802e-01,GAS
5,6,419.226072,-0.000282,-0.000168,2.662067e-01,CARBON
6,7,849.872855,-0.001939,-0.006202,7.811906e-01,GAS
7,8,825.083975,-0.000267,0.000279,3.731606e-01,GAS
8,9,774.995508,-0.000093,-0.000449,1.969420e-01,CARBON
9,10,668.960253,-0.000140,-0.000489,4.821915e-01,CARBON


In [35]:
df_daily_eq.groupby('Combustible')[['phi']].mean()

,phi
Combustible,
ACPM,890.036969
CARBON,328.097040
GAS,898.218699
MEZCLA GAS - CARBÓN,520.568058


In [ ]:
df_resultados_individuales.columns

In [ ]:
df_daily_eq=df_daily_eq.merge(df_resultados_individuales[['id_planta', 'beta_S1', 'beta_S2', 'beta_S3']], on='id_planta')

In [ ]:
df_daily_eq.columns

In [ ]:


import numpy as np

df_daily_eq['phi19'] = df_daily_eq.apply(
    lambda row: (
        row['beta_S1']
        + row['beta_S2'] * np.asarray(row['qmin'])
        + (row['beta_S3'] / 2) * np.asarray(row['S3_vector_it'])
    ),
    axis=1
)

In [ ]:
df_daily_eq

In [ ]:
df_daily_eq[['phi', 'phi19']]

In [ ]:
df_daily_eq[df_daily_eq['CodigoPlanta']=='TYP3']['phi'].describe()

In [ ]:
df_daily_eq[df_daily_eq['CodigoPlanta']=='TYP3']['phi19']

In [ ]:
df_daily_eq['phi19'].tolist()

In [ ]:
[]